# 05 — Final Colab Runs: LLMLingua-2, Two-Turn History, Judge Calibration, Error Analysis

This notebook is the reproducibility entry point for the final report. It runs the experiments that should not be executed on a local laptop: real LLMLingua-2, synthetic two-turn history pruning, LLM-as-Judge calibration, and qualitative error-analysis exports.

Expected runtime: use a Colab GPU runtime. Mistral-7B full generation/judging usually needs a high-memory GPU or quantization adjustments.


In [ ]:
# Runtime setup
# In Colab: Runtime -> Change runtime type -> GPU

# Clone or update repo
import os
import sys
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Mrtuzy/CENG467_Final.git"
REPO_DIR = Path("/content/CENG467_Final")

if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
    print(f"Removing non-git directory: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print(f"Working directory: {Path.cwd()}")


In [ ]:
# Dependencies
# llmlingua is intentionally installed here so S1 cannot silently be reported as RECOMP.

packages = [
    "datasets",
    "rank_bm25",
    "tqdm",
    "pandas",
    "matplotlib",
    "scikit-learn",
    "transformers",
    "accelerate",
    "sentence-transformers",
    "llmlingua",
    "bert-score",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

try:
    from llmlingua import PromptCompressor
    print("LLMLingua import OK. Real S1 run is enabled.")
except Exception as exc:
    raise RuntimeError("LLMLingua is not available; do not report S1 as a real LLMLingua-2 result.") from exc


In [ ]:
# Configuration

MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
N_EVAL = 50          # Use 50 for progress/final sanity; change to 200 for full final run.
EVAL_START = 600     # Keeps eval split away from LNN training cells.
K_RETRIEVE = 5
MAX_TOKENS = 512
RESULTS_DIR = Path("experiments/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RUN_FULL_LLM = True  # Set False for pruning-only smoke tests.

print({
    "model": MODEL,
    "n_eval": N_EVAL,
    "eval_start": EVAL_START,
    "k_retrieve": K_RETRIEVE,
    "max_tokens": MAX_TOKENS,
    "run_full_llm": RUN_FULL_LLM,
})


In [ ]:
# Load HotpotQA directly in Colab and convert to project format

from datasets import load_dataset

raw = load_dataset("hotpot_qa", "distractor", trust_remote_code=True)
val_data = raw["validation"]

def _supporting_indices(sf):
    # HotpotQA fields differ slightly across local helpers/notebooks.
    return sf.get("sent_id", sf.get("sent_idx", []))

def convert_sample(sample):
    return {
        "id": sample["id"],
        "question": sample["question"],
        "answer": sample["answer"],
        "context": list(zip(sample["context"]["title"], sample["context"]["sentences"])),
        "supporting_facts": {
            "title": sample["supporting_facts"]["title"],
            "sent_id": _supporting_indices(sample["supporting_facts"]),
        },
        "type": sample.get("type"),
        "level": sample.get("level"),
    }

all_samples = [convert_sample(val_data[i]) for i in range(len(val_data))]
eval_samples = all_samples[EVAL_START:EVAL_START + N_EVAL]
calibration_samples = all_samples[EVAL_START + N_EVAL:EVAL_START + N_EVAL + 50]

print(f"Evaluation samples: {len(eval_samples)}")
print(f"Calibration samples: {len(calibration_samples)}")


In [ ]:
# Imports and helpers

import json
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import cohen_kappa_score

from src.retrieval.bm25_retriever import BM25Retriever
from src.pruning import PRUNER_REGISTRY
from src.pruning.history_pruning import HistoryPruner
from src.pruning.combined import CombinedPruner
from src.generation.generator import RAGGenerator
from src.judge.judge import LLMJudge
from src.evaluation.metrics import exact_match, token_f1, compression_ratio
from src.evaluation.coverage_noise import coverage_noise_report
from src.utils.io import save_jsonl


def supporting_sentences(sample):
    context_map = {title: sents for title, sents in sample["context"]}
    sf = sample.get("supporting_facts", {})
    sents = []
    for title, idx in zip(sf.get("title", []), sf.get("sent_id", [])):
        if title in context_map and idx < len(context_map[title]):
            sents.append(context_map[title][idx])
    return sents


def synthetic_two_turn_history(sample):
    evidence = supporting_sentences(sample)
    if evidence:
        return ["Earlier turn evidence: " + " ".join(evidence[:2])]
    return ["Earlier turn question: " + sample["question"]]


def retrieve(sample, k=K_RETRIEVE):
    passages = [" ".join(sents) for _, sents in sample["context"]]
    titles = [title for title, _ in sample["context"]]
    retriever = BM25Retriever()
    retriever.index(passages, titles)
    return retriever.retrieve(sample["question"], k=k)


def build_pruner(pruner_name, sample=None, two_turn=False):
    if pruner_name == "history_pruning" and two_turn:
        return HistoryPruner(history=synthetic_two_turn_history(sample))
    if pruner_name == "combined" and two_turn:
        # CombinedPruner constructor may vary; fall back to setting nested history if available.
        pruner = CombinedPruner()
        if hasattr(pruner, "history_pruner"):
            pruner.history_pruner.history = synthetic_two_turn_history(sample)
        return pruner
    return PRUNER_REGISTRY[pruner_name]()


def was_llmlingua_fallback(pruner_name, pruner):
    if pruner_name != "llmlingua2":
        return False
    return bool(getattr(pruner, "fallback_used", getattr(pruner, "_compressor", None) is None))


In [ ]:
# Load generator and judge once, then share model weights

if RUN_FULL_LLM:
    generator = RAGGenerator(MODEL)
    generator.load_model()
    judge = LLMJudge(MODEL)
    judge._model = generator._model
    judge._tokenizer = generator._tokenizer
else:
    generator = RAGGenerator("dry_run")
    judge = LLMJudge("dry_run")

print("Generator and judge ready.")


In [ ]:
# Final gap-closing runs
# These are the rows needed to replace the weak spots in the report.

RUNS = [
    {"name": "recomp", "label": "B3: RECOMP", "two_turn": False},
    {"name": "llmlingua2", "label": "S1: LLMLingua-2 real", "two_turn": False},
    {"name": "history_pruning", "label": "S3: History Pruning two-turn", "two_turn": True},
    {"name": "combined", "label": "S4: Combined two-turn", "two_turn": True},
]

all_aggregates = []

for run in RUNS:
    pruner_name = run["name"]
    output_path = RESULTS_DIR / f"{pruner_name}_{N_EVAL}samples{'_twoturn' if run['two_turn'] else '_final'}.jsonl"
    rows = []
    metrics = {"faithfulness": [], "em": [], "f1": [], "compression_ratio": [], "latency_s": [], "coverage": [], "noise_ratio": [], "coverage_noise_f1": []}
    fallback_count = 0

    print(f"\n=== {run['label']} ===")
    for sample in tqdm(eval_samples):
        retrieved = retrieve(sample)
        input_tokens = sum(len(p["passage"].split()) for p in retrieved)
        pruner = build_pruner(pruner_name, sample=sample, two_turn=run["two_turn"])

        t0 = time.time()
        pruned = pruner.prune(retrieved, sample["question"], max_tokens=MAX_TOKENS)
        prune_latency = time.time() - t0

        llmlingua_fallback = was_llmlingua_fallback(pruner_name, pruner)
        fallback_count += int(llmlingua_fallback)
        output_tokens = sum(len(p["passage"].split()) for p in pruned)

        gen_result = generator.generate(sample["question"], pruned)
        judge_result = judge.score(gen_result["answer"], pruned)
        cn = coverage_noise_report(pruned, sample)

        row = {
            "sample_id": sample["id"],
            "question": sample["question"],
            "gold_answer": sample["answer"],
            "generated_answer": gen_result["answer"],
            "pruner": pruner_name,
            "run_label": run["label"],
            "two_turn_history": run["two_turn"],
            "synthetic_history": synthetic_two_turn_history(sample) if run["two_turn"] else [],
            "llmlingua_fallback": llmlingua_fallback,
            "faithfulness": judge_result["faithfulness"],
            "n_claims": judge_result["n_claims"],
            "claims": judge_result.get("claims", []),
            "em": exact_match(gen_result["answer"], sample["answer"]),
            "f1": token_f1(gen_result["answer"], sample["answer"]),
            "compression_ratio": compression_ratio(input_tokens, output_tokens),
            "latency_s": gen_result["latency_s"] + prune_latency,
            "coverage": cn["coverage"],
            "noise_ratio": cn["noise_ratio"],
            "coverage_noise_f1": cn["coverage_noise_f1"],
            "retrieved": retrieved,
            "pruned_context": pruned,
            "supporting_sentences": supporting_sentences(sample),
        }
        rows.append(row)
        for key in metrics:
            metrics[key].append(row[key])

    save_jsonl(rows, str(output_path))
    aggregate = {
        "run_label": run["label"],
        "pruner": pruner_name,
        "n_samples": len(rows),
        "output_path": str(output_path),
        "llmlingua_fallback_count": fallback_count,
        **{f"{key}_mean": float(np.mean(vals)) for key, vals in metrics.items()},
    }
    all_aggregates.append(aggregate)
    print(json.dumps(aggregate, indent=2))

summary_path = RESULTS_DIR / "final_gap_closing_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(all_aggregates, f, indent=2)
print(f"Saved summary: {summary_path}")


In [ ]:
# LLM-as-Judge calibration
# Step 1: Run this cell once to create an annotation template.
# Step 2: Fill human_faithfulness manually for at least 20 rows, preferably 50.
# Step 3: Re-run the next cell to compute Cohen's kappa.

CAL_DIR = Path("data/judge_calibration")
CAL_DIR.mkdir(parents=True, exist_ok=True)
TEMPLATE_PATH = CAL_DIR / "human_labels_template.jsonl"
HUMAN_LABELS_PATH = CAL_DIR / "human_labels.jsonl"

calibration_template = []
for sample in calibration_samples:
    retrieved = retrieve(sample)
    context = "\n\n".join(p["passage"] for p in retrieved)
    calibration_template.append({
        "sample_id": sample["id"],
        "question": sample["question"],
        "gold_answer": sample["answer"],
        "context": context[:4000],
        "human_faithfulness": None,
        "notes": "Fill with 1.0 if supported, 0.0 if unsupported, or fractional if partially supported.",
    })

with open(TEMPLATE_PATH, "w", encoding="utf-8") as f:
    for row in calibration_template:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Template written to {TEMPLATE_PATH}")
print(f"After annotation, save as {HUMAN_LABELS_PATH}")


In [ ]:
# Compute judge calibration against human labels

if not HUMAN_LABELS_PATH.exists():
    print(f"Human labels not found yet: {HUMAN_LABELS_PATH}")
    print("Annotate human_labels_template.jsonl, save it as human_labels.jsonl, then re-run this cell.")
else:
    human_rows = []
    with open(HUMAN_LABELS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if row.get("human_faithfulness") is not None:
                human_rows.append(row)

    sample_by_id = {s["id"]: s for s in calibration_samples}
    judge_scores = []
    human_scores = []
    details = []

    for row in tqdm(human_rows, desc="Calibration"):
        sample = sample_by_id[row["sample_id"]]
        retrieved = retrieve(sample)
        # Calibrate the judge on the gold answer against retrieved evidence.
        judge_result = judge.score(sample["answer"], retrieved)
        judge_scores.append(judge_result["faithfulness"])
        human_scores.append(float(row["human_faithfulness"]))
        details.append({**row, "judge_faithfulness": judge_result["faithfulness"], "claims": judge_result.get("claims", [])})

    judge_binary = [1 if s > 0.5 else 0 for s in judge_scores]
    human_binary = [1 if s > 0.5 else 0 for s in human_scores]
    kappa = cohen_kappa_score(human_binary, judge_binary)

    calibration_result = {
        "n_labels": len(human_rows),
        "cohen_kappa": float(kappa),
        "judge_scores": judge_scores,
        "human_scores": human_scores,
    }

    with open(CAL_DIR / "calibration_results.json", "w", encoding="utf-8") as f:
        json.dump(calibration_result, f, indent=2)
    with open(CAL_DIR / "calibration_details.jsonl", "w", encoding="utf-8") as f:
        for row in details:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(json.dumps(calibration_result, indent=2))


In [ ]:
# Error-analysis export
# Picks high-signal examples for manual discussion in the final report.

candidate_files = list(RESULTS_DIR.glob("*50samples*.jsonl")) + list(RESULTS_DIR.glob("*200samples*.jsonl"))
print("Available result files:")
for path in candidate_files:
    print(" -", path)

examples = []
for path in candidate_files:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            label = None
            if row.get("coverage", 1.0) < 1.0:
                label = "pruning_error_support_removed"
            elif row.get("faithfulness", 1.0) < 0.5:
                label = "generation_or_judge_faithfulness_failure"
            elif row.get("em", 1.0) == 0 and row.get("f1", 1.0) < 0.3:
                label = "verbose_or_non_exact_answer"
            if label:
                examples.append({
                    "error_type": label,
                    "source_file": str(path),
                    "pruner": row.get("run_label", row.get("pruner")),
                    "sample_id": row.get("sample_id"),
                    "question": row.get("question"),
                    "gold_answer": row.get("gold_answer"),
                    "generated_answer": row.get("generated_answer"),
                    "faithfulness": row.get("faithfulness"),
                    "em": row.get("em"),
                    "f1": row.get("f1"),
                    "coverage": row.get("coverage"),
                    "noise_ratio": row.get("noise_ratio"),
                    "supporting_sentences": row.get("supporting_sentences", [])[:2],
                    "notes": "Fill this manually before copying into the report.",
                })

# Keep a compact set with varied error types/pruners.
selected = []
seen = set()
for ex in examples:
    key = (ex["error_type"], ex["pruner"])
    if key not in seen:
        selected.append(ex)
        seen.add(key)
    if len(selected) >= 12:
        break

out_path = RESULTS_DIR / "error_analysis_candidates.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for row in selected:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

pd.DataFrame(selected)[["error_type", "pruner", "question", "gold_answer", "generated_answer", "faithfulness", "coverage"]].head(12)


In [ ]:
# Report-ready LaTeX table rows

summary_path = RESULTS_DIR / "final_gap_closing_summary.json"
if summary_path.exists():
    with open(summary_path, "r", encoding="utf-8") as f:
        summary = json.load(f)

    for row in summary:
        print(
            f"{row['run_label']} & "
            f"{row['faithfulness_mean']:.3f} & "
            f"{row['em_mean']:.3f} & "
            f"{row['f1_mean']:.3f} & "
            f"{row['compression_ratio_mean']:.3f} & "
            f"{row['latency_s_mean']:.2f} \\" 
        )
else:
    print("Run the final gap-closing experiments first.")
